### Transformada Discreta de Fourier

### 1. Simplificação com $W_N$

Para simplificar a notação, define-se o **fator de giro**:

$$\boxed{W_N = e^{-j\frac{2\pi}{N}}}$$

Com isso, as fórmulas da DFT ficam:

$$\boxed{X(k) = \sum_{n=0}^{N-1} x(n)\, W_N^{kn}, \quad k = 0, 1, \ldots, N-1}$$


### 2. Representação Matricial da DFT

Escrevendo todos os N valores $X(k)$ explicitamente:

$$X(0) = x(0)W_N^{0\cdot0} + x(1)W_N^{0\cdot1} + x(2)W_N^{0\cdot2} + \cdots + x(N-1)W_N^{0\cdot(N-1)}$$
$$X(1) = x(0)W_N^{1\cdot0} + x(1)W_N^{1\cdot1} + x(2)W_N^{1\cdot2} + \cdots + x(N-1)W_N^{1\cdot(N-1)}$$
$$\vdots$$
$$X(N-1) = x(0)W_N^{(N-1)\cdot0} + x(1)W_N^{(N-1)\cdot1} + \cdots + x(N-1)W_N^{(N-1)\cdot(N-1)}$$

Resumindo:

$$\bar{X} = \overline{W}_N \cdot \bar{x}$$

onde a **matriz DFT** $\overline{W}_N$ é:

$$\overline{W}_N = \begin{bmatrix}
W_N^{0} & W_N^{0} & W_N^{0} & \cdots & W_N^{0} \\
W_N^{0} & W_N^{1} & W_N^{2} & \cdots & W_N^{N-1} \\
W_N^{0} & W_N^{2} & W_N^{4} & \cdots & W_N^{2(N-1)} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
W_N^{0} & W_N^{N-1} & W_N^{2(N-1)} & \cdots & W_N^{(N-1)^2}
\end{bmatrix}$$


### 3. Implementação — DFT Matricial

Formúla:

$$\bar{X} = \overline{W}_N \cdot \bar{x}$$

In [1]:
# Importando bibliotecas.
import numpy as np
import matplotlib.pyplot as plt
import time

### 3.1 Construindo a Matriz DFT $\overline{W}_N$

O elemento $(k, n)$ da matriz é $W_N^{kn} = e^{-j\frac{2\pi}{N}kn}$


In [2]:
def construir_matriz_dft(N):
    """
    Constrói a matriz DFT de N pontos.
    
    Parâmetros:
        N : int — número de pontos da DFT
    
    Retorna:
        W : matriz complexa NxN
    """
    
    # Fator de giro.
    W_N = np.exp(-1j * 2 * np.pi / N)
    
    # Constrói a matriz usando:
    # k = linha (0..N-1), n = coluna (0..N-1)
    k = np.arange(N).reshape(N, 1)   # vetor coluna: [[0],[1],...,[N-1]]
    n = np.arange(N).reshape(1, N)   # vetor linha:  [[0, 1, ..., N-1]]
    
    # W[k,n] = W_N^(k*n) = e^(-j*2pi/N * k * n)
    W = W_N ** (k * n)
    
    return W


### 3.2 DFT Matricial: $\bar{X} = \overline{W}_N \cdot \bar{x}$


In [3]:
def dft_matricial(x, N=None):
    """
    Calcula a DFT usando produto matricial.
    
    Parâmetros:
        x : array: sinal de entrada (comprimento L)
        N : int: número de pontos da DFT (N >= L)
    
    Retorna:
        X : array complexo de N pontos
    """
    
    x = np.array(x, dtype=complex)
    L = len(x)
    
    if N is None:
        N = L
    
    if N < L:
        raise ValueError(f"N={N} deve ser >= L={L} para evitar time aliasing!")
    
    # Zero-padding se necessário
    if N > L:
        x = np.concatenate([x, np.zeros(N - L, dtype=complex)])
    
    # Constrói a matriz DFT
    W = construir_matriz_dft(N)
    
    # Produto matricial: X = W @ x
    X = W @ x
    
    return X


### 4. Custo Computacional da DFT Matricial

Para calcular um único valor $X(k)$, o somatório requer:
- N multiplicações complexas
- N-1 adições complexas

Igualmente a DFT direta:

$$\text{Multiplicações: } N^2 \qquad \text{Adições: } (N-1)N \longrightarrow
\text{Complexidade: } \mathcal{O}(N^2)$$

In [4]:
def medir_tempo_dft_matricial(x, N):
    """
    Recebe um sinal x qualquer e mede o tempo da DFT matricial.
    """
    x = np.array(x, dtype=complex)

    # Nossa DFT
    t0 = time.time()
    X_matricial = dft_matricial(x, N)
    t_matricial = (time.time() - t0) * 1000

    print(f"Tamanho do sinal:  N = {N}")
    print(f"DFT Matricial:        {t_matricial:.4f} ms")

    return X_matricial

In [7]:
print("=== Implementação da DFT Matricial ===")
print("Digite os valores das amostras separados por espaço (ex: 0 1 2 3):")

L = int(input("Quantidade de amostras do sinal (L): "))
N = int(input("Número de pontos da DFT (N >= L):   "))

np.random.seed(42)
x = np.random.randn(L)

X = medir_tempo_dft_matricial(x, N=N)

print(f"\n=== Saída da DFT Matricial ({N} pontos) ===")

if len(X) > 10:
    t = 10
else:
    t = len(X)
    
for i in range(t):
    print(f"  X[{i}] = {X[i]:.2f}")

=== Implementação da DFT Matricial ===
Digite os valores das amostras separados por espaço (ex: 0 1 2 3):
Tamanho do sinal:  N = 1024
DFT Matricial:        126.5726 ms

=== Saída da DFT Matricial (1024 pontos) ===
  X[0] = 30.08+0.00j
  X[1] = 29.15+8.67j
  X[2] = -30.28+34.26j
  X[3] = -14.00+22.97j
  X[4] = -34.86-9.42j
  X[5] = 25.65+13.12j
  X[6] = -1.41-2.54j
  X[7] = 27.36-17.36j
  X[8] = 8.44+20.10j
  X[9] = 9.46+29.13j


In [6]:
import json

tamanhos = [16, 32, 64, 128, 256, 512, 1024, 2048]
REPETICOES = 5

tempos_ms = []

for N in tamanhos:
    np.random.seed(42)
    x = np.random.randn(N)
    medicoes = []
    for _ in range(REPETICOES):
        t0 = time.time()
        dft_matricial(x)
        medicoes.append((time.time() - t0) * 1000)
    media = round(float(np.mean(medicoes)), 4)
    tempos_ms.append(media)
    print(f"N = {N:5d} → {media:.4f} ms")

metricas = {
    "nome"        : "DFT Matricial",
    "tamanhos"    : tamanhos,
    "tempos_ms"   : tempos_ms
}

with open("dft_matricial_metrics.json", "w") as f:
    json.dump(metricas, f, indent=2)

print("\nSalvo em dft_matricial_metrics.json")

N =    16 → 0.0334 ms
N =    32 → 0.1200 ms
N =    64 → 4.0268 ms
N =   128 → 3.9847 ms
N =   256 → 13.6905 ms
N =   512 → 54.2675 ms
N =  1024 → 163.9609 ms
N =  2048 → 543.2142 ms

Salvo em dft_matricial_metrics.json
